# UChicago Trading Competition 2022 - Case 3: Black-Litterman rebalancing

Daily allocator over 9 assets (A-I) that blends three analysts' price forecasts
with market-cap equilibrium returns via the Black-Litterman model.

Fixes vs. the original Colab version:
- Runs locally or in Colab (no hard-coded Drive paths).
- Cells in dependency order; state is reset per backtest.
- Prior covariance comes from realized returns (analyst forecasts are piecewise
  constant, so their covariance was near-singular).
- Equilibrium returns and mean-variance weights use the risk-aversion `delta`
  (the original passed `tau` in its place and multiplied by an arbitrary 2).
- `np.linalg.pinv` replaces the `lstsq(..., P)` fallbacks.
- Weights are normalized to gross exposure 1 (sum of |w| = 1).
- The Sharpe filter compares portfolios on a time series of realized returns,
  not across the 9 assets of a single day.
- Vectorized returns/covariance (the per-day Python loops were O(n^2)).
- The backtest computes a real annualized Sharpe ratio and compares it to 1/n.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:  # optional: only when running inside Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/case3data')
except ImportError:
    DATA_DIR = Path('case3data')

N_ASSETS = 9
TAU = 0.05            # Black-Litterman uncertainty scaling of the prior
DELTA = 2.4           # risk-aversion parameter (2.15 - 2.65)
SHARPE_LOOKBACK = 20  # days of realized returns used by the Sharpe filter
MAX_HISTORY = 9       # accepted weight vectors averaged into the held portfolio
P = np.eye(N_ASSETS)  # picking matrix: one absolute view per asset

shares = pd.read_csv(DATA_DIR / 'Shares Outstanding.csv', index_col=0, encoding='utf-8-sig')
SHARES = shares.iloc[0].to_numpy(dtype=float)


In [2]:
def log_returns(prices):
    """(T, N) prices -> (T-1, N) log returns."""
    return np.diff(np.log(prices), axis=0)


def covariance(returns):
    """Sample covariance (N, N) of a (T, N) return array."""
    return np.cov(returns, rowvar=False)


def implied_market_returns(prices, sigma, delta, shares):
    """Equilibrium returns Pi = delta * Sigma * w_mkt (market-cap weights)."""
    mcap = prices * shares
    w_mkt = mcap / mcap.sum()
    return delta * sigma @ w_mkt


def black_litterman(sigma, pi, P, Q, tau):
    """Posterior expected returns with Omega = tau * P Sigma P^T."""
    tau_sigma_inv = np.linalg.pinv(tau * sigma)
    omega_inv = np.linalg.pinv(tau * P @ sigma @ P.T)
    A = tau_sigma_inv + P.T @ omega_inv @ P
    b = tau_sigma_inv @ pi + P.T @ omega_inv @ Q
    return np.linalg.pinv(A) @ b


def mean_variance_weights(sigma, mu, delta):
    """Unconstrained mean-variance weights w = (delta * Sigma)^-1 mu."""
    return np.linalg.pinv(delta * sigma) @ mu


def normalize_gross(w):
    """Scale so that sum(|w|) = 1; equal weights if w is all zeros."""
    gross = np.abs(w).sum()
    if not np.isfinite(gross) or gross == 0:
        return np.repeat(1 / len(w), len(w))
    return w / gross


def sharpe(returns):
    """Mean / std of a return series (not annualized); -inf if undefined."""
    sd = returns.std()
    return returns.mean() / sd if sd > 0 else -np.inf


In [3]:
class BLAllocator:
    def __init__(self):
        self.prices = []
        self.preds = [[], [], []]
        self.accepted = [np.repeat(1 / N_ASSETS, N_ASSETS)]

    def held(self):
        return normalize_gross(np.mean(self.accepted, axis=0))

    def allocate(self, asset_prices, *predictions):
        self.prices.append(np.asarray(asset_prices, dtype=float))
        for hist, p in zip(self.preds, predictions):
            hist.append(np.asarray(p, dtype=float))

        # need at least 2 returns to estimate a covariance
        if len(self.prices) < 3:
            return np.repeat(1 / N_ASSETS, N_ASSETS)

        prices = np.array(self.prices)
        ret_real = log_returns(prices)
        sigma = covariance(ret_real)
        pi = implied_market_returns(prices[-1], sigma, DELTA, SHARES)

        candidate = np.zeros(N_ASSETS)
        for hist in self.preds:
            Q = log_returns(np.array(hist[-2:]))[0]  # latest change in the analyst's forecast
            mu = black_litterman(sigma, pi, P, Q, TAU)
            candidate += normalize_gross(mean_variance_weights(sigma, mu, DELTA))
        candidate = normalize_gross(candidate)

        # Sharpe filter on a time series of recent realized returns
        window = ret_real[-SHARPE_LOOKBACK:]
        if sharpe(window @ candidate) > sharpe(window @ self.held()):
            if len(self.accepted) >= MAX_HISTORY:
                self.accepted = [np.repeat(1 / N_ASSETS, N_ASSETS)]
            self.accepted.append(candidate)
        return self.held()


_allocator = BLAllocator()


def allocate_portfolio(asset_prices, asset_price_predictions_1,
                       asset_price_predictions_2, asset_price_predictions_3):
    """Competition entry point: returns 9 weights with sum(|w|) = 1."""
    return _allocator.allocate(asset_prices, asset_price_predictions_1,
                               asset_price_predictions_2, asset_price_predictions_3)


In [4]:
def load(name):
    return pd.read_csv(DATA_DIR / name, index_col=0).to_numpy(dtype=float)

actual = load('Actual Testing Data.csv')
pred = [load(f'Predicted Testing Data Analyst {i}.csv') for i in (1, 2, 3)]


def backtest(alloc_fn, n_days=None):
    n = len(actual) if n_days is None else n_days
    simple_ret = actual[1:n] / actual[:n - 1] - 1
    weights = np.array([alloc_fn(actual[t], pred[0][t], pred[1][t], pred[2][t])
                        for t in range(n - 1)])
    port = (weights * simple_ret).sum(axis=1)  # weights at t earn the t -> t+1 return
    return {
        'annualized_sharpe': np.sqrt(252) * port.mean() / port.std(),
        'cumulative_return': np.prod(1 + port) - 1,
        'avg_gross_exposure': np.abs(weights).sum(axis=1).mean(),
    }


_allocator = BLAllocator()  # fresh state for every run
results = pd.DataFrame({
    'Black-Litterman': backtest(allocate_portfolio),
    '1/n': backtest(lambda prices, *_: np.repeat(1 / N_ASSETS, N_ASSETS)),
})
results


,Black-Litterman,1/n
annualized_sharpe,1.444128,1.175046
cumulative_return,33.010939,10.842140
avg_gross_exposure,1.000000,1.000000
